# Level 1: five-fold cross-validation

Run the same MLP experiment five times, holding out a different development fold each time. Every call creates a fresh model and fits feature/target scaling on its four training folds.

**Fixed settings:** level 1; seed 42; architecture `50 -> 32 -> 16 -> 2`; batch size 64; AdamW, learning rate `1e-3`, weight decay `1e-4`; 20 epochs. These settings are currently defined in `mlp/experiment.py`. Each run restores its lowest-validation-MSE checkpoint.

**Run:** choose the project `.venv/bin/python` kernel and run all cells from top to bottom. After editing imported modules, restart the kernel. Rerunning the experiment cell replaces `fold_results` with five fresh runs.

**Scope:** only level 1 development folds are loaded. This assesses the current workflow across development splits. Each held-out fold also selects its checkpoint, and fold 5 has already informed development, so these are development results rather than an unbiased final test estimate. Test data remains untouched. Levels 2 and 3 are not used here.


## 1. Locate the project and import helpers

The setup works from either the project root, `mlp/`, or `mlp/level1/`. We use the existing experiment function instead of duplicating the training workflow. NumPy provides summaries; no new dependencies are needed.

In [1]:
from pathlib import Path
import sys
import contextlib
import io

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "mlp" / "data.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook inside the cosmoreg project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from mlp import data
from mlp.experiment import run_experiment

LEVEL_ROOT = PROJECT_ROOT / "mlp" / "level1"
SPECTRA_ROOT = PROJECT_ROOT / "powerspectrum"
LEVEL = 1
FOLDS = range(1, 6)
print("Project:", PROJECT_ROOT)


Project: /home/meatpigeon/Personal/classes/csci494/cosmoreg


## 2. Check the development folds

Count maps and independent simulations, and verify that the five folds share no simulation IDs. We load only `fold1` through `fold5`, never `test`.

The expected level 1 development set has 54 simulations and 810 maps. Four folds have 11 simulations each, while fold 5 has 10. We keep these counts to weight the final summary fairly.

In [2]:
fold_counts = {}
seen_ids = set()

for fold in FOLDS:
    X, y, simulation_ids = data.load_fold(SPECTRA_ROOT, LEVEL, fold)
    ids = set(np.unique(simulation_ids).tolist())
    assert not seen_ids.intersection(ids), "Simulation IDs overlap across folds."
    seen_ids.update(ids)
    assert X.shape == (len(y), 50)
    assert y.shape == (len(X), 2)
    fold_counts[fold] = len(ids)
    print(f"Fold {fold}: {len(X)} maps, {len(ids)} simulations")

assert len(seen_ids) == 54
print("Development simulations:", len(seen_ids))


Fold 1: 165 maps, 11 simulations
Fold 2: 165 maps, 11 simulations
Fold 3: 165 maps, 11 simulations
Fold 4: 165 maps, 11 simulations
Fold 5: 150 maps, 10 simulations
Development simulations: 54


## 3. Run five fresh experiments

For validation fold 1, training uses folds 2–5; for validation fold 2, training uses folds 1, 3, 4, and 5; and so on. The experiment checks training/validation separation for every call.

`fold_results` stores the metrics, histories, best models, and training-fitted scalers. To keep the output readable, each fold's detailed training log is captured in `fold_logs`; the cell prints a short completion summary. For example, inspect `print(fold_logs[1])` to see fold 1's epoch losses.

This cell trains 100 epochs in total: 20 for each of five independent runs.

In [3]:
fold_results = []
fold_logs = {}

for fold in FOLDS:
    print(f"Running level {LEVEL}, validation fold {fold}...", flush=True)
    log = io.StringIO()
    with contextlib.redirect_stdout(log):
        results = run_experiment(SPECTRA_ROOT, LEVEL, fold)
    fold_logs[fold] = log.getvalue()
    assert results["validation_fold"] == fold
    assert len(results["train_history"]) == len(results["val_history"]) == 20
    assert np.isclose(results["best_val_loss"], min(results["val_history"]))
    fold_results.append(results)
    print(
        f"  Best epoch {results['best_epoch']:2d} | "
        f"validation MSE {results['best_val_loss']:.4f} | "
        f"baseline MSE {results['baseline_loss']:.4f}"
    )

assert len(fold_results) == 5


Running level 1, validation fold 1...
  Best epoch 14 | validation MSE 0.4187 | baseline MSE 0.7113
Running level 1, validation fold 2...
  Best epoch  5 | validation MSE 0.7496 | baseline MSE 1.1382
Running level 1, validation fold 3...
  Best epoch 19 | validation MSE 1.0920 | baseline MSE 1.3640
Running level 1, validation fold 4...
  Best epoch 11 | validation MSE 0.5809 | baseline MSE 0.8746
Running level 1, validation fold 5...
  Best epoch 18 | validation MSE 1.1517 | baseline MSE 1.2424


## 4. Compare simulation-level errors for every fold

All errors below are in original target units. Each simulation contributes one averaged prediction. The baseline uses that run's training target means, so it changes with the training split.

```text
RMSE improvement (%) = 100 * (1 - model_RMSE / baseline_RMSE)
```

Positive improvement means lower RMSE than the baseline. RMSE and MAE can disagree, so inspect both. `simulation_metrics` arrays follow `[Omega_m, sigma_8]`.

In [4]:
target_names = fold_results[0]["target_names"]

for i, target in enumerate(target_names):
    print(f"\n{target}: simulation-level validation")
    print("Fold  Sims  Epoch  MLP RMSE  Base RMSE  MLP MAE  Base MAE  RMSE gain")
    for results in fold_results:
        fold = results["validation_fold"]
        metrics = results["simulation_metrics"]
        base_rmse = metrics["baseline_rmse"][i]
        gain = 100 * (1 - metrics["rmse"][i] / base_rmse) if base_rmse > 0 else float("nan")
        print(
            f"{fold:4d}  {fold_counts[fold]:4d}  {results['best_epoch']:5d}  "
            f"{metrics['rmse'][i]:8.4f}  {base_rmse:9.4f}  "
            f"{metrics['mae'][i]:7.4f}  {metrics['baseline_mae'][i]:8.4f}  "
            f"{gain:8.1f}%"
        )



Omega_m: simulation-level validation
Fold  Sims  Epoch  MLP RMSE  Base RMSE  MLP MAE  Base MAE  RMSE gain
   1    11     14    0.0712     0.0907   0.0597    0.0739      21.5%
   2    11      5    0.0807     0.1151   0.0672    0.0960      29.9%
   3    11     19    0.0979     0.1140   0.0922    0.1039      14.1%
   4    11     11    0.0787     0.1019   0.0592    0.0798      22.8%
   5    10     18    0.1155     0.1203   0.1066    0.1060       4.0%

sigma_8: simulation-level validation
Fold  Sims  Epoch  MLP RMSE  Base RMSE  MLP MAE  Base MAE  RMSE gain
   1    11     14    0.0592     0.0991   0.0514    0.0817      40.2%
   2    11      5    0.0962     0.1138   0.0795    0.0987      15.5%
   3    11     19    0.1082     0.1320   0.0938    0.1140      18.0%
   4    11     11    0.0753     0.1045   0.0604    0.0864      27.9%
   5    10     18    0.1039     0.1176   0.0926    0.1064      11.7%


## 5. Summarize fold variation and pooled development errors

The mean and sample standard deviation of fold RMSE describe variation across the five splits. The folds' training sets overlap, so this standard deviation is **not** an uncertainty estimate from five independent experiments.

For the pooled summary, weight each fold by its number of validation simulations. Each of the 54 development simulations is held out once. Pool squared errors before taking the square root; averaging fold RMSE values is a different statistic.

```text
pooled_RMSE = sqrt(sum(fold_count * fold_RMSE**2) / sum(fold_count))
pooled_MAE = sum(fold_count * fold_MAE) / sum(fold_count)
```

The baseline is also pooled using the same weights. `cv_summary` holds these calculations for later inspection. These are results of validation-selected checkpoints, not final test results.

In [5]:
counts = np.array([fold_counts[r["validation_fold"]] for r in fold_results])
cv_summary = {}

for key in ("rmse", "mae", "baseline_rmse", "baseline_mae"):
    values = np.stack([r["simulation_metrics"][key] for r in fold_results])
    assert values.shape == (5, 2) and np.isfinite(values).all()
    if key.endswith("rmse"):
        cv_summary[f"pooled_{key}"] = np.sqrt(np.average(values**2, axis=0, weights=counts))
    else:
        cv_summary[f"pooled_{key}"] = np.average(values, axis=0, weights=counts)
    cv_summary[f"fold_mean_{key}"] = values.mean(axis=0)
    cv_summary[f"fold_std_{key}"] = values.std(axis=0, ddof=1)

for i, target in enumerate(target_names):
    wins = sum(r["simulation_metrics"]["rmse"][i] < r["simulation_metrics"]["baseline_rmse"][i]
               for r in fold_results)
    base = cv_summary["pooled_baseline_rmse"][i]
    gain = 100 * (1 - cv_summary["pooled_rmse"][i] / base) if base > 0 else float("nan")
    print(f"\n{target}:")
    print(f"  Fold MLP RMSE mean +/- SD: {cv_summary['fold_mean_rmse'][i]:.4f} +/- {cv_summary['fold_std_rmse'][i]:.4f}")
    print(f"  Lower RMSE than baseline in {wins}/5 folds")
    print(f"  Pooled MLP / baseline RMSE: {cv_summary['pooled_rmse'][i]:.4f} / {base:.4f} ({gain:.1f}% improvement)")
    print(f"  Pooled MLP / baseline MAE: {cv_summary['pooled_mae'][i]:.4f} / {cv_summary['pooled_baseline_mae'][i]:.4f}")

print("\nBest epochs:", [r["best_epoch"] for r in fold_results])
print("Validation simulations represented:", int(counts.sum()))



Omega_m:
  Fold MLP RMSE mean +/- SD: 0.0888 +/- 0.0179
  Lower RMSE than baseline in 5/5 folds
  Pooled MLP / baseline RMSE: 0.0897 / 0.1087 (17.5% improvement)
  Pooled MLP / baseline MAE: 0.0764 / 0.0916

sigma_8:
  Fold MLP RMSE mean +/- SD: 0.0886 +/- 0.0207
  Lower RMSE than baseline in 5/5 folds
  Pooled MLP / baseline RMSE: 0.0902 / 0.1139 (20.8% improvement)
  Pooled MLP / baseline MAE: 0.0752 / 0.0973

Best epochs: [14, 5, 19, 11, 18]
Validation simulations represented: 54


## What to inspect next

Does improvement hold across folds and across both error measures? How much do best epochs vary, and how often is the best epoch at the 20-epoch limit? These results inform the next development decision; do not use the test set to choose settings.

A later experiment can assess seed variation or levels 2 and 3 using separate runs. This notebook performs only the fixed-seed level 1 comparison. Models, scalers, and logs are available in memory after running; persistent checkpoint saving is a separate step.
